# HealthConnect Week 8 final analytics

**Wilson Moses | AnalystLab Africa Experience Lab**

This notebook builds the final descriptive decision-support view from the approved synthetic appointment file. It preserves the raw source, separates cancellations, and avoids causal claims. No Week 7 Data Analytics results file was available in the reviewed materials.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np
PACKAGE=Path.cwd() if (Path.cwd()/'data/raw').is_dir() else Path.cwd().parent
SOURCE=PACKAGE/'data/raw/HealthConnect_Appointment_Data.csv'
df=pd.read_csv(SOURCE,keep_default_na=False)
print('Rows and columns:',df.shape)
print('Source SHA-256:',hashlib.sha256(SOURCE.read_bytes()).hexdigest())
print('Outcomes:',df.appointment_outcome.value_counts().to_dict())

Rows and columns: (5000, 18)
Source SHA-256: 1f2c40ae14833b46fe6e1a353446a0e038f19ae06434b37d98545b3e28697137
Outcomes: {'No-Show': 2423, 'Attended': 2314, 'Cancelled': 263}


## Data preparation

Blank distance and waiting-time values are parsed as missing values for checks. Dates and lead days are parsed for validation. The raw source remains unchanged.

In [2]:
analysis=df.copy()
for col in ['distance_to_clinic_km','waiting_time_minutes']:
    analysis[col]=pd.to_numeric(analysis[col].replace('',np.nan),errors='coerce')
for col in ['booking_date','appointment_date']:
    analysis[col]=pd.to_datetime(analysis[col],errors='coerce')
analysis['booking_lead_days']=pd.to_numeric(analysis['booking_lead_days'],errors='coerce')
analysis['eligible']=analysis.appointment_outcome.isin(['Attended','No-Show'])
analysis['no_show']=analysis.appointment_outcome.eq('No-Show')
completed=analysis[analysis.eligible]
print('Completed appointments:',len(completed))
print('Distinct patients:',analysis.patient_id.nunique())
print('Repeated patient IDs:',int((analysis.patient_id.value_counts()>1).sum()))
print('Missing distance:',int(analysis.distance_to_clinic_km.isna().sum()))
print('Missing waiting time:',int(analysis.waiting_time_minutes.isna().sum()))

Completed appointments: 4737
Distinct patients: 1696
Repeated patient IDs: 1394
Missing distance: 90
Missing waiting time: 60


## KPI definitions

Attendance rate = Attended / (Attended + No-Show). No-show rate = No-Show / (Attended + No-Show). Cancellation rate = Cancelled / all appointment records. Reminder coverage = Reminder Sent = Yes / all appointment records.

In [3]:
attended=int(analysis.appointment_outcome.eq('Attended').sum())
no_shows=int(analysis.appointment_outcome.eq('No-Show').sum())
cancelled=int(analysis.appointment_outcome.eq('Cancelled').sum())
completed_n=attended+no_shows
kpis=pd.DataFrame([['Attendance rate',attended,completed_n,attended/completed_n],['No-show rate',no_shows,completed_n,no_shows/completed_n],['Cancellation rate',cancelled,len(analysis),cancelled/len(analysis)],['Reminder coverage',int(analysis.reminder_sent.eq('Yes').sum()),len(analysis),analysis.reminder_sent.eq('Yes').mean()]],columns=['metric','numerator','denominator','rate'])
print(kpis.to_string(index=False,formatters={'rate':lambda x:f'{x:.1%}'}))

           metric  numerator  denominator  rate
  Attendance rate       2314         4737 48.8%
     No-show rate       2423         4737 51.2%
Cancellation rate        263         5000  5.3%
Reminder coverage       3634         5000 72.7%


## Lead-time pattern

Lead-time groups are 0-3, 4-7, 8-14, 15-30 and 31+ days. Patient IDs repeat, so rows are not independent people. These results are descriptive and do not establish cause.

In [4]:
analysis['lead_band']=pd.cut(analysis.booking_lead_days,[-1,3,7,14,30,np.inf],labels=['0-3 days','4-7 days','8-14 days','15-30 days','31+ days'])
completed=analysis[analysis.eligible]
lead_results=completed.groupby('lead_band',observed=False).no_show.agg(['sum','count','mean']).reset_index()
lead_results.columns=['lead_band','no_show_count','completed_appointments','no_show_rate']
print(lead_results.to_string(index=False,formatters={'no_show_rate':lambda x:f'{x:.1%}'}))
print('This is a descriptive association. It does not show that longer lead time causes a missed appointment.')

 lead_band  no_show_count  completed_appointments no_show_rate
  0-3 days             79                     297        26.6%
  4-7 days             99                     307        32.2%
 8-14 days            202                     574        35.2%
15-30 days            576                    1265        45.5%
  31+ days           1467                    2294        63.9%
This is a descriptive association. It does not show that longer lead time causes a missed appointment.


## Reminder and appointment type

The reminder comparison describes recorded groups. It does not estimate reminder effectiveness. Appointment type results describe service mix and should not be read as causal.

In [5]:
reminder_results=completed.groupby('reminder_sent').no_show.agg(['sum','count','mean']).reset_index()
type_results=completed.groupby('appointment_type').no_show.agg(['sum','count','mean']).sort_values('mean',ascending=False).reset_index()
print('Reminder status'); print(reminder_results.to_string(index=False,formatters={'mean':lambda x:f'{x:.1%}'}))
print('Appointment type'); print(type_results.to_string(index=False,formatters={'mean':lambda x:f'{x:.1%}'}))
print('Reminder differences are observational. Channel selection and delivery are not controlled here.')

Reminder status
reminder_sent  sum  count  mean
           No  702   1285 54.6%
          Yes 1721   3452 49.9%
Appointment type
       appointment_type  sum  count  mean
              Follow-up  728   1344 54.2%
        Diagnostic Test  295    567 52.0%
Specialist Consultation  427    846 50.5%
   General Consultation  973   1980 49.1%
Reminder differences are observational. Channel selection and delivery are not controlled here.


## Validation

The source checks cover unique appointment IDs, duplicate rows, lead days, appointment weekday, and previous appointment counts. Limited missing distance and waiting-time values are documented.

In [6]:
lead_from_dates=(analysis.appointment_date-analysis.booking_date).dt.days
weekday_from_dates=analysis.appointment_date.dt.day_name()
checks={'unique appointment IDs':analysis.appointment_id.is_unique,'no exact duplicate rows':not analysis.duplicated().any(),'lead days match dates':(lead_from_dates==analysis.booking_lead_days).all(),'weekday matches appointment date':(weekday_from_dates==analysis.appointment_day).all(),'prior no-shows do not exceed prior appointments':(analysis.previous_no_shows.astype(int)<=analysis.previous_appointments.astype(int)).all()}
for name,passed in checks.items(): print(('PASS' if passed else 'REVIEW')+' | '+name)
assert all(checks.values())

PASS | unique appointment IDs
PASS | no exact duplicate rows
PASS | lead days match dates
PASS | weekday matches appointment date
PASS | prior no-shows do not exceed prior appointments
